# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

**Desarrollo — Jhonn Saeteros.** Se reutilizan los 18 libros de NLTK Gutenberg y la normalización de las prácticas anteriores: minúsculas, conservación de letras y separación por espacios, sin eliminar stopwords ni aplicar stemming. Ejecute las celdas en orden con Python 3. Dependencias: `nltk`, `numpy`, `pandas`, `scipy`, `scikit-learn` e `ipykernel`.

Se usan $TF=f(t,d)$ e $IDF_{TFIDF}=\ln(N/df(t))+1$. Para BM25 se elige la variante positiva:

$$IDF_{BM25}(t)=\ln\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right)$$

$$BM25(q,d)=\sum_{t\in\operatorname{set}(q)\cap V}IDF_{BM25}(t)\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/\operatorname{avgdl})}$$

Cada término distinto de la consulta se cuenta una vez. La variante de IDF se documenta en [Lucene BM25Similarity](https://lucene.apache.org/core/9_12_1/core/org/apache/lucene/search/similarities/BM25Similarity.html). Se utilizó inteligencia artificial como apoyo para implementar, documentar y verificar el procedimiento.

In [1]:
import re
from pathlib import Path
from collections import Counter
from itertools import combinations
import nltk
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from nltk.corpus import gutenberg
from scipy.sparse import issparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
pd.set_option('display.max_colwidth', 180)

try:
    file_ids = sorted(gutenberg.fileids())
except LookupError:
    ruta_datos = Path.cwd() / 'nltk_data'
    nltk.data.path.insert(0, str(ruta_datos))
    if not nltk.download('gutenberg', download_dir=str(ruta_datos), quiet=True):
        raise RuntimeError('No se pudo descargar Gutenberg. Revise la conexión.')
    file_ids = sorted(gutenberg.fileids())

# Misma configuración de las prácticas 02 y 03; no se eliminan stopwords.
def normalizar_y_tokenizar(texto):
    texto = texto.lower()
    texto = re.sub(r'[^a-záéíóúñü]+', ' ', texto)
    return texto.split()

def obtener_titulo(texto):
    encabezado = re.search(r'\[([^\]]+)\]', texto)
    return encabezado.group(1).strip() if encabezado else texto.splitlines()[0].strip()

raw_docs = {fid: gutenberg.raw(fid) for fid in file_ids}
corpus = pd.DataFrame({
    'doc_id': [str(fid) for fid in file_ids],
    'Título': [obtener_titulo(raw_docs[fid]) for fid in file_ids],
    'Texto': [raw_docs[fid] for fid in file_ids]
})
corpus['tokens'] = corpus['Texto'].map(normalizar_y_tokenizar)
corpus['longitud'] = corpus['tokens'].map(len)
display(corpus)
N = len(corpus)
doc_ids = corpus['doc_id'].tolist()
titulos = corpus['Título'].tolist()
tokens_corpus = corpus['tokens'].tolist()
tf_documentos = [Counter(tokens) for tokens in tokens_corpus]
longitudes = corpus['longitud'].to_numpy(dtype=float)
avgdl = float(longitudes.mean())
df_corpus = Counter()
for tokens in tokens_corpus:
    df_corpus.update(set(tokens))
vocabulario = sorted(df_corpus)
indice_terminos = {t: i for i, t in enumerate(vocabulario)}
idf_tfidf = {t: np.log(N / df_corpus[t]) + 1 for t in vocabulario}
# Variante positiva de IDF BM25: evita puntuaciones negativas en términos comunes.
idf_bm25 = {t: np.log1p((N - df_corpus[t] + 0.5) / (df_corpus[t] + 0.5))
            for t in vocabulario}
estadisticas_terminos = pd.DataFrame({
    'Término': vocabulario,
    'DF': [df_corpus[t] for t in vocabulario],
    'IDF TF-IDF': [idf_tfidf[t] for t in vocabulario],
    'IDF BM25': [idf_bm25[t] for t in vocabulario]
})
display(estadisticas_terminos)
print(f'Documentos: {N} | Vocabulario: {len(vocabulario):,}')
print(f'Longitudes (tokens): mínima={longitudes.min():,.0f}, '
      f'media={avgdl:,.2f}, máxima={longitudes.max():,.0f}')
assert N == 18 and all(isinstance(fid, str) for fid in doc_ids)

,doc_id,Título,Texto,tokens,longitud
0,austen-emma.txt,Emma by Jane Austen 1816,"[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAPTER I\n\n\nEmma Woodhouse, handsome, clever, and rich, with a comfortable home\nand happy disposition, seemed to unite some of the...","[emma, by, jane, austen, volume, i, chapter, i, emma, woodhouse, handsome, clever, and, rich, with, a, comfortable, home, and, happy, disposition, seemed, to, unite, some, of, ...",161975
1,austen-persuasion.txt,Persuasion by Jane Austen 1818,"[Persuasion by Jane Austen 1818]\n\n\nChapter 1\n\n\nSir Walter Elliot, of Kellynch Hall, in Somersetshire, was a man who,\nfor his own amusement, never took up any book but th...","[persuasion, by, jane, austen, chapter, sir, walter, elliot, of, kellynch, hall, in, somersetshire, was, a, man, who, for, his, own, amusement, never, took, up, any, book, but,...",84123
2,austen-sense.txt,Sense and Sensibility by Jane Austen 1811,"[Sense and Sensibility by Jane Austen 1811]\n\nCHAPTER 1\n\n\nThe family of Dashwood had long been settled in Sussex.\nTheir estate was large, and their residence was at Norlan...","[sense, and, sensibility, by, jane, austen, chapter, the, family, of, dashwood, had, long, been, settled, in, sussex, their, estate, was, large, and, their, residence, was, at,...",120734
3,bible-kjv.txt,The King James Bible,[The King James Bible]\n\nThe Old Testament of the King James Bible\n\nThe First Book of Moses: Called Genesis\n\n\n1:1 In the beginning God created the heaven and the earth.\...,"[the, king, james, bible, the, old, testament, of, the, king, james, bible, the, first, book, of, moses, called, genesis, in, the, beginning, god, created, the, heaven, and, th...",791842
4,blake-poems.txt,Poems by William Blake 1789,"[Poems by William Blake 1789]\n\n \nSONGS OF INNOCENCE AND OF EXPERIENCE\nand THE BOOK of THEL\n\n\n SONGS OF INNOCENCE\n \n \n INTRODUCTION\n \n Piping down the valleys wild,\...","[poems, by, william, blake, songs, of, innocence, and, of, experience, and, the, book, of, thel, songs, of, innocence, introduction, piping, down, the, valleys, wild, piping, s...",6934
5,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryant 1918,"[Stories to Tell to Children by Sara Cone Bryant 1918] \r\n\r\n\r\nTWO LITTLE RIDDLES IN RHYME\r\n\r\n\r\n There's a garden that I ken,\r\n Full of little gentlemen;\r\...","[stories, to, tell, to, children, by, sara, cone, bryant, two, little, riddles, in, rhyme, there, s, a, garden, that, i, ken, full, of, little, gentlemen, little, caps, of, blu...",46697
6,burgess-busterbrown.txt,The Adventures of Buster Bear by Thornton W. Burgess 1920,[The Adventures of Buster Bear by Thornton W. Burgess 1920]\r\n\r\nI\r\n\r\nBUSTER BEAR GOES FISHING\r\n\r\n\r\nBuster Bear yawned as he lay on his comfortable bed of leaves an...,"[the, adventures, of, buster, bear, by, thornton, w, burgess, i, buster, bear, goes, fishing, buster, bear, yawned, as, he, lay, on, his, comfortable, bed, of, leaves, and, wat...",16353
7,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carroll 1865,"[Alice's Adventures in Wonderland by Lewis Carroll 1865]\n\nCHAPTER I. Down the Rabbit-Hole\n\nAlice was beginning to get very tired of sitting by her sister on the\nbank, and ...","[alice, s, adventures, in, wonderland, by, lewis, carroll, chapter, i, down, the, rabbit, hole, alice, was, beginning, to, get, very, tired, of, sitting, by, her, sister, on, t...",27335
8,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,[The Ball and The Cross by G.K. Chesterton 1909]\n\n\nI. A DISCUSSION SOMEWHAT IN THE AIR\n\nThe flying ship of Professor Lucifer sang through the skies like\na silver arrow; t...,"[the, ball, and, the, cross, by, g, k, chesterton, i, a, discussion, somewhat, in, the, air, the, flying, ship, of, professor, lucifer, sang, through, the, skies, like, a, silv...",82788
9,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton 1914,"[The Wisdom of Father Brow

,Término,DF,IDF TF-IDF,IDF BM25
0,a,18,1.000000,0.026668
1,aaron,2,3.197225,2.028148
2,aaronites,1,3.890372,2.538974
3,ab,1,3.890372,2.538974
4,aback,2,3.197225,2.028148
...,...,...,...,...
41504,zur,1,3.890372,2.538974
41505,zuriel,1,3.890372,2.538974
41506,zurishaddai,1,3.890372,2.538974
41507,zuyder,1,3.890372,2.538974


Documentos: 18 | Vocabulario: 41,509
Longitudes (tokens): mínima=6,934, media=118,671.06, máxima=791,842


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
consultas = [
    ('Q1', 'whale sea ship', 'Encontrar relatos de navegación y caza de ballenas.'),
    ('Q2', 'love marriage family', 'Encontrar historias de amor, matrimonio y familia.'),
    ('Q3', 'god heaven faith', 'Encontrar textos sobre religión y vida espiritual.'),
    ('Q4', 'king murder ghost', 'Encontrar relatos de reyes, asesinatos y fantasmas.'),
    ('Q5', 'the and', 'Explorar coincidencias con dos palabras funcionales muy frecuentes.'),
    ('Q6', 'leviathan', 'Encontrar menciones del leviatán con una consulta específica.'),
    ('Q7', 'harpooneer whale ocean voyage ship sea',
     'Encontrar relatos de arponeros y travesías marítimas mediante una consulta larga.'),
    ('Q8', 'zzzzinexistentterm', 'Comprobar la búsqueda de un término ajeno al vocabulario.')
]
tokens_consultas = {qid: normalizar_y_tokenizar(texto) for qid, texto, _ in consultas}
display(pd.DataFrame(consultas, columns=['Consulta', 'Texto', 'Necesidad de información']))
filas_df = []
for qid, texto, necesidad in consultas:
    for t in dict.fromkeys(tokens_consultas[qid]):
        filas_df.append({'Consulta': qid, 'Término': t, 'DF': df_corpus[t],
                         'Proporción de documentos': df_corpus[t] / N,
                         'IDF BM25': idf_bm25.get(t, 0.0)})
display(pd.DataFrame(filas_df))
display(Markdown(
    f"**Justificación con DF.** `the` aparece en {df_corpus['the']}/{N} libros y "
    f"`and` en {df_corpus['and']}/{N}: representan términos muy frecuentes. "
    f"`leviathan` aparece en {df_corpus['leviathan']}/{N} y `harpooneer` en "
    f"{df_corpus['harpooneer']}/{N}: permiten examinar términos menos frecuentes. "
    "Las consultas tienen entre uno y seis tokens; Q1–Q4 cubren necesidades distintas, "
    "Q5 comprueba palabras comunes, Q6 especificidad, Q7 mayor longitud y Q8 ausencia total."
))

def identidad(tokens):
    return tokens

# Igual TF-IDF que en la práctica 03: TF directo, ln(N/DF)+1, sin suavizado.
# Se conserva la representación dispersa; no se convierte la matriz a densa.
vectorizador = TfidfVectorizer(analyzer=identidad, vocabulary=indice_terminos,
                              use_idf=True, smooth_idf=False,
                              sublinear_tf=False, norm=None, dtype=np.float64)
matriz_tfidf = vectorizador.fit_transform(tokens_corpus).tocsr()
assert issparse(matriz_tfidf)
np.testing.assert_allclose(vectorizador.idf_, [idf_tfidf[t] for t in vocabulario])

def aportes_bm25(tokens_consulta, i_documento, k1=1.2, b=0.75):
    filas = []
    # Un aporte por término distinto, sin factor extra de frecuencia de consulta.
    for t in sorted(set(tokens_consulta)):
        frecuencia = tf_documentos[i_documento][t]
        ajuste = 1 - b + b * longitudes[i_documento] / avgdl
        factor_tf = frecuencia * (k1 + 1) / (frecuencia + k1 * ajuste)
        idf = idf_bm25.get(t, 0.0)
        filas.append({'Término': t, 'f(t,d)': frecuencia, 'df(t)': df_corpus[t],
                      'IDF BM25': idf, 'Ajuste de longitud': ajuste,
                      'Factor TF saturado': factor_tf, 'Aporte BM25': idf * factor_tf})
    return pd.DataFrame(filas)

def calcular_bm25(tokens_consulta, k1=1.2, b=0.75):
    scores = np.zeros(N)
    ajuste = 1 - b + b * longitudes / avgdl
    for t in sorted(set(tokens_consulta)):
        if t not in idf_bm25:
            continue
        frecuencias = np.array([tf[t] for tf in tf_documentos], dtype=float)
        scores += idf_bm25[t] * frecuencias * (k1 + 1) / (frecuencias + k1 * ajuste)
    return scores

def ranking(scores, limite=10):
    indices = [i for i in range(N) if scores[i] > 0]
    indices.sort(key=lambda i: (-float(scores[i]), doc_ids[i]))
    return [{'Posición': pos, 'doc_id': doc_ids[i], 'Título': titulos[i],
             'Score': float(scores[i])}
            for pos, i in enumerate(indices[:limite], 1)]

estadisticas_antes = (dict(df_corpus), dict(idf_bm25), longitudes.copy())
resultados = {}
for qid, texto, _ in consultas:
    vector_q = vectorizador.transform([tokens_consultas[qid]])
    scores_tf = cosine_similarity(vector_q, matriz_tfidf).ravel()
    scores_bm = calcular_bm25(tokens_consultas[qid])
    resultados[qid] = {'tfidf': ranking(scores_tf), 'bm25': ranking(scores_bm),
                       'scores_tf': scores_tf, 'scores_bm': scores_bm}
    display(Markdown(f'**{qid}: {texto}**'))
    display(Markdown('TF-IDF + coseno'))
    display(pd.DataFrame(resultados[qid]['tfidf'],
                         columns=['Posición', 'doc_id', 'Título', 'Score']))
    display(Markdown('BM25: k₁ = 1.2; b = 0.75'))
    display(pd.DataFrame(resultados[qid]['bm25'],
                         columns=['Posición', 'doc_id', 'Título', 'Score']))
    assert np.all(np.isfinite(scores_bm)) and np.all(scores_bm >= 0)
assert resultados['Q8']['tfidf'] == resultados['Q8']['bm25'] == []
assert estadisticas_antes[0] == dict(df_corpus)
assert estadisticas_antes[1] == dict(idf_bm25)
np.testing.assert_array_equal(estadisticas_antes[2], longitudes)
print('Estadísticas de colección conservadas; Q8 produce dos listas vacías.')

,Consulta,Texto,Necesidad de información
0,Q1,whale sea ship,Encontrar relatos de navegación y caza de ballenas.
1,Q2,love marriage family,"Encontrar historias de amor, matrimonio y familia."
2,Q3,god heaven faith,Encontrar textos sobre religión y vida espiritual.
3,Q4,king murder ghost,"Encontrar relatos de reyes, asesinatos y fantasmas."
4,Q5,the and,Explorar coincidencias con dos palabras funcionales muy frecuentes.
5,Q6,leviathan,Encontrar menciones del leviatán con una consulta específica.
6,Q7,harpooneer whale ocean voyage ship sea,Encontrar relatos de arponeros y travesías marítimas mediante una consulta larga.
7,Q8,zzzzinexistentterm,Comprobar la búsqueda de un término ajeno al vocabulario.


,Consulta,Término,DF,Proporción de documentos,IDF BM25
0,Q1,whale,6,0.333333,1.072637
1,Q1,sea,16,0.888889,0.141079
2,Q1,ship,13,0.722222,0.341749
3,Q2,love,15,0.833333,0.203599
4,Q2,marriage,12,0.666667,0.418710
5,Q2,family,13,0.722222,0.341749
6,Q3,god,16,0.888889,0.141079
7,Q3,heaven,13,0.722222,0.341749
8,Q3,faith,15,0.833333,0.203599
9,Q4,king,15,0.833333,0.203599


**Justificación con DF.** `the` aparece en 18/18 libros y `and` en 18/18: representan términos muy frecuentes. `leviathan` aparece en 3/18 y `harpooneer` en 2/18: permiten examinar términos menos frecuentes. Las consultas tienen entre uno y seis tokens; Q1–Q4 cubren necesidades distintas, Q5 comprueba palabras comunes, Q6 especificidad, Q7 mayor longitud y Q8 ausencia total.

**Q1: whale sea ship**

TF-IDF + coseno

,Posición,doc_id,Título,Score
0,1,melville-moby_dick.txt,Moby Dick by Herman Melville 1851,0.117301
1,2,whitman-leaves.txt,Leaves of Grass by Walt Whitman 1855,0.010530
2,3,bryant-stories.txt,Stories to Tell to Children by Sara Cone Bryant 1918,0.007388
3,4,chesterton-ball.txt,The Ball and The Cross by G.K. Chesterton 1909,0.006308
4,5,chesterton-brown.txt,The Wisdom of Father Brown by G. K. Chesterton 1914,0.005437
5,6,milton-paradise.txt,Paradise Lost by John Milton 1667,0.004032
6,7,chesterton-thursday.txt,The Man Who Was Thursday by G. K. Chesterton 1908,0.002881
7,8,shakespeare-hamlet.txt,The Tragedie of Hamlet by William Shakespeare 1599,0.002638
8,9,bible-kjv.txt,The King James Bible,0.002267
9,10,carroll-alice.txt,Alice's Adventures in Wonderland by Lewis Carroll 1865,0.002142


... (34 outputs left out)

,Posición,doc_id,Título,Score


BM25: k₁ = 1.2; b = 0.75

,Posición,doc_id,Título,Score


Estadísticas de colección conservadas; Q8 produce dos listas vacías.


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [3]:
resumen = []
for qid, texto, _ in consultas:
    tf, bm = resultados[qid]['tfidf'], resultados[qid]['bm25']
    filas = []
    for p in range(max(len(tf), len(bm))):
        a = tf[p] if p < len(tf) else None
        b = bm[p] if p < len(bm) else None
        filas.append({'Posición': p + 1,
                      'ID y título TF-IDF': f"{a['doc_id']} | {a['Título']}" if a else '—',
                      'Coseno': a['Score'] if a else np.nan,
                      'ID y título BM25': f"{b['doc_id']} | {b['Título']}" if b else '—',
                      'Score BM25': b['Score'] if b else np.nan})
    display(Markdown(f'**Comparación {qid}: {texto}**'))
    display(pd.DataFrame(filas, columns=['Posición', 'ID y título TF-IDF', 'Coseno',
                                        'ID y título BM25', 'Score BM25']))
    ids_tf = {fila['doc_id'] for fila in tf}
    ids_bm = {fila['doc_id'] for fila in bm}
    resumen.append({'Consulta': qid, 'Tamaño TF-IDF': len(tf), 'Tamaño BM25': len(bm),
                    'IDs compartidos top-10': len(ids_tf & ids_bm),
                    'Primero TF-IDF': tf[0]['doc_id'] if tf else 'Lista vacía',
                    'Primero BM25': bm[0]['doc_id'] if bm else 'Lista vacía',
                    'Mismo orden': [r['doc_id'] for r in tf] == [r['doc_id'] for r in bm]})
display(pd.DataFrame(resumen))

# Se busca un par con inversión de orden dentro de los top-10 de Q2.
consulta_analisis = 'Q2'
resultado = resultados[consulta_analisis]
pos_tf = {r['doc_id']: r['Posición'] for r in resultado['tfidf']}
pos_bm = {r['doc_id']: r['Posición'] for r in resultado['bm25']}
comunes = sorted(set(pos_tf) & set(pos_bm))
pares = [(a, b) for a, b in combinations(comunes, 2)
         if (pos_tf[a] - pos_tf[b]) * (pos_bm[a] - pos_bm[b]) < 0]
assert pares, 'Se esperaba un par de documentos con inversión de orden en Q2.'
doc_a, doc_b = max(pares, key=lambda par: abs(pos_tf[par[0]] - pos_tf[par[1]])
                   + abs(pos_bm[par[0]] - pos_bm[par[1]]))
indices_analisis = [doc_ids.index(doc_a), doc_ids.index(doc_b)]
display(Markdown(f'**Análisis de cambio de posición: {consulta_analisis}, '
                 f"consulta `{dict((q, t) for q, t, _ in consultas)[consulta_analisis]}`.**"))
filas_docs, detalles = [], []
for i in indices_analisis:
    fid = doc_ids[i]
    filas_docs.append({'doc_id': fid, '|d|': int(longitudes[i]), 'avgdl': avgdl,
                       '|d| / avgdl': longitudes[i] / avgdl,
                       'Posición TF-IDF': pos_tf[fid], 'Coseno': resultado['scores_tf'][i],
                       'Posición BM25': pos_bm[fid], 'Score BM25': resultado['scores_bm'][i]})
    aportes = aportes_bm25(tokens_consultas[consulta_analisis], i)
    np.testing.assert_allclose(aportes['Aporte BM25'].sum(), resultado['scores_bm'][i])
    aportes.insert(0, 'doc_id', fid)
    aportes.insert(1, '|d|', int(longitudes[i]))
    detalles.append(aportes)
display(pd.DataFrame(filas_docs))
display(pd.concat(detalles, ignore_index=True))

# Contribuciones al coseno: pesos del documento y consulta divididos por sus normas.
q_vector = vectorizador.transform([tokens_consultas[consulta_analisis]])
norma_q = np.sqrt(float(q_vector.multiply(q_vector).sum()))
filas_coseno = []
for i in indices_analisis:
    d_vector = matriz_tfidf.getrow(i)
    norma_d = np.sqrt(float(d_vector.multiply(d_vector).sum()))
    for t in sorted(set(tokens_consultas[consulta_analisis])):
        j = indice_terminos[t]
        aporte = float(q_vector[0, j] * d_vector[0, j]) / (norma_q * norma_d)
        filas_coseno.append({'doc_id': doc_ids[i], 'Término': t,
                             'Norma documento TF-IDF': norma_d,
                             'Aporte al coseno': aporte})
    suma = sum(r['Aporte al coseno'] for r in filas_coseno if r['doc_id'] == doc_ids[i])
    np.testing.assert_allclose(suma, resultado['scores_tf'][i])
display(pd.DataFrame(filas_coseno))

ganador_bm = max(indices_analisis, key=lambda i: resultado['scores_bm'][i])
otro = next(i for i in indices_analisis if i != ganador_bm)
aportes_g = aportes_bm25(tokens_consultas[consulta_analisis], ganador_bm).set_index('Término')
aportes_o = aportes_bm25(tokens_consultas[consulta_analisis], otro).set_index('Término')
ventajas = aportes_g['Aporte BM25'] - aportes_o['Aporte BM25']
termino_decisivo = ventajas.idxmax()
display(Markdown(
    f"**Causa concreta de la inversión.** `{doc_ids[ganador_bm]}` supera a "
    f"`{doc_ids[otro]}` en BM25 por "
    f"{resultado['scores_bm'][ganador_bm] - resultado['scores_bm'][otro]:.6f} puntos. "
    f"La mayor ventaja por término procede de `{termino_decisivo}`: sus frecuencias "
    f"son {int(aportes_g.loc[termino_decisivo, 'f(t,d)'])} y "
    f"{int(aportes_o.loc[termino_decisivo, 'f(t,d)'])}, con aportes "
    f"{aportes_g.loc[termino_decisivo, 'Aporte BM25']:.6f} y "
    f"{aportes_o.loc[termino_decisivo, 'Aporte BM25']:.6f}. "
    f"Su DF es {df_corpus[termino_decisivo]} y su IDF BM25 es "
    f"{idf_bm25[termino_decisivo]:.6f}. En TF-IDF, las normas de los libros son "
    f"{np.sqrt(float(matriz_tfidf.getrow(ganador_bm).multiply(matriz_tfidf.getrow(ganador_bm)).sum())):.3f} y "
    f"{np.sqrt(float(matriz_tfidf.getrow(otro).multiply(matriz_tfidf.getrow(otro)).sum())):.3f}; "
    'la normalización reduce de forma distinta los productos compartidos y deja el orden inverso.'
))

for i in indices_analisis:
    fid = doc_ids[i]
    aportes = aportes_bm25(tokens_consultas[consulta_analisis], i)
    dominante = aportes.loc[aportes['Aporte BM25'].idxmax()]
    display(Markdown(
        f"- **{fid}** tiene {int(longitudes[i]):,} tokens ({longitudes[i]/avgdl:.3f} "
        f"veces la media). Pasa de la posición **{pos_tf[fid]}** de TF-IDF "
        f"(coseno **{resultado['scores_tf'][i]:.6f}**) a la posición **{pos_bm[fid]}** "
        f"de BM25 (score **{resultado['scores_bm'][i]:.6f}**). Su mayor aporte BM25 "
        f"es `{dominante['Término']}`: f={int(dominante['f(t,d)'])}, "
        f"df={int(dominante['df(t)'])}, aporte={dominante['Aporte BM25']:.6f}."
    ))
display(Markdown(
    '**Explicación del cambio.** La tabla de coseno muestra cómo los pesos compartidos ' 
    'se dividen por la norma completa de cada libro; sus aportes suman el coseno ' 
    'que determina ese ranking. En BM25 los aportes suman su propio score. Una frecuencia ' 
    'alta se satura hacia el factor k₁+1=2.2, en vez de crecer linealmente. ' 
    'La columna de ajuste de longitud es menor que 1 en los libros más cortos que la ' 
    'media y mayor que 1 en los más largos: modifica el denominador y, con ello, cada ' 
    'aporte. Un df menor produce un IDF mayor para ambos documentos. Estas diferencias ' 
    'explican la inversión observada; las magnitudes de coseno y BM25 se interpretan ' 
    'dentro de cada método, pues tienen escalas distintas.'
))

**Comparación Q1: whale sea ship**

,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,0.117301,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,3.414106
1,2,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,0.010530,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,3.232519
2,3,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,0.007388,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,2.933924
3,4,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,0.006308,shakespeare-hamlet.txt | The Tragedie of Hamlet by William Shakespeare 1599,2.745454
4,5,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,0.005437,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,2.651290
5,6,milton-paradise.txt | Paradise Lost by John Milton 1667,0.004032,bible-kjv.txt | The King James Bible,1.756872
6,7,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,0.002881,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,1.023872
7,8,shakespeare-hamlet.txt | The Tragedie of Hamlet by William Shakespeare 1599,0.002638,austen-persuasion.txt | Persuasion by Jane Austen 1818,0.947304
8,9,bible-kjv.txt | The King James Bible,0.002267,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,0.851800
9,10,carroll-alice.txt | Alice's Adventures in Wonderland by Lewis Carroll 1865,0.002142,shakespeare-macbeth.txt | The Tragedie of Macbeth by William Shakespeare 1603,0.793421


**Comparación Q2: love marriage family**

,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,blake-poems.txt | Poems by William Blake 1789,0.026155,austen-sense.txt | Sense and Sensibility by Jane Austen 1811,2.078348
1,2,austen-persuasion.txt | Persuasion by Jane Austen 1818,0.013830,austen-persuasion.txt | Persuasion by Jane Austen 1818,2.071504
2,3,austen-sense.txt | Sense and Sensibility by Jane Austen 1811,0.013032,austen-emma.txt | Emma by Jane Austen 1816,2.061976
3,4,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,0.012918,bible-kjv.txt | The King James Bible,1.845840
4,5,austen-emma.txt | Emma by Jane Austen 1816,0.010585,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,1.842628
5,6,milton-paradise.txt | Paradise Lost by John Milton 1667,0.009994,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,1.840525
6,7,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,0.005700,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,1.670932
7,8,"edgeworth-parents.txt | The Parent's Assistant, by Maria Edgeworth",0.005226,milton-paradise.txt | Paradise Lost by John Milton 1667,1.544266
8,9,bible-kjv.txt | The King James Bible,0.002942,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,1.526304
9,10,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,0.002764,"edgeworth-parents.txt | The Parent's Assistant, by Maria Edgeworth",1.161775


**Comparación Q3: god heaven faith**

,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,milton-paradise.txt | Paradise Lost by John Milton 1667,0.078211,milton-paradise.txt | Paradise Lost by John Milton 1667,1.496070
1,2,bible-kjv.txt | The King James Bible,0.032087,bible-kjv.txt | The King James Bible,1.490508
2,3,blake-poems.txt | Poems by William Blake 1789,0.020790,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,1.461438
3,4,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,0.014485,chesterton-ball.txt | The Ball and The Cross by G.K. Chesterton 1909,1.460335
4,5,shakespeare-hamlet.txt | The Tragedie of Hamlet by William Shakespeare 1599,0.008507,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,1.426747
5,6,shakespeare-macbeth.txt | The Tragedie of Macbeth by William Shakespeare 1603,0.007295,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,1.395483
6,7,whitman-leaves.txt | Leaves of Grass by Walt Whitman 1855,0.006791,"edgeworth-parents.txt | The Parent's Assistant, by Maria Edgeworth",1.316335
7,8,melville-moby_dick.txt | Moby Dick by Herman Melville 1851,0.006647,chesterton-brown.txt | The Wisdom of Father Brown by G. K. Chesterton 1914,1.305737
8,9,chesterton-thursday.txt | The Man Who Was Thursday by G. K. Chesterton 1908,0.005792,austen-sense.txt | Sense and Sensibility by Jane Austen 1811,1.289787
9,10,bryant-stories.txt | Stories to Tell to Children by Sara Cone Bryant 1918,0.003715,austen-persuasion.txt | Persuasion by Jane Austen 1818,1.287992


... (15 outputs left out)

**Causa concreta de la inversión.** `bible-kjv.txt` supera a `edgeworth-parents.txt` en BM25 por 0.684065 puntos. La mayor ventaja por término procede de `marriage`: sus frecuencias son 19 y 0, con aportes 0.691637 y 0.000000. Su DF es 12 y su IDF BM25 es 0.418710. En TF-IDF, las normas de los libros son 103315.258 y 15591.663; la normalización reduce de forma distinta los productos compartidos y deja el orden inverso.

- **bible-kjv.txt** tiene 791,842 tokens (6.673 veces la media). Pasa de la posición **9** de TF-IDF (coseno **0.002942**) a la posición **4** de BM25 (score **1.845840**). Su mayor aporte BM25 es `family`: f=123, df=13, aporte=0.715186.

- **edgeworth-parents.txt** tiene 170,786 tokens (1.439 veces la media). Pasa de la posición **8** de TF-IDF (coseno **0.005226**) a la posición **10** de BM25 (score **1.161775**). Su mayor aporte BM25 es `family`: f=39, df=13, aporte=0.722304.

**Explicación del cambio.** La tabla de coseno muestra cómo los pesos compartidos se dividen por la norma completa de cada libro; sus aportes suman el coseno que determina ese ranking. En BM25 los aportes suman su propio score. Una frecuencia alta se satura hacia el factor k₁+1=2.2, en vez de crecer linealmente. La columna de ajuste de longitud es menor que 1 en los libros más cortos que la media y mayor que 1 en los más largos: modifica el denominador y, con ello, cada aporte. Un df menor produce un IDF mayor para ambos documentos. Estas diferencias explican la inversión observada; las magnitudes de coseno y BM25 se interpretan dentro de cada método, pues tienen escalas distintas.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [4]:
# Se conserva Q2 y se examinan los mismos documentos en las tres configuraciones.
# Se agregan el libro más corto y el más largo con coincidencias para contrastar longitudes.
tokens_parametros = tokens_consultas[consulta_analisis]
con_coincidencias = [i for i in range(N) if resultados[consulta_analisis]['scores_bm'][i] > 0]
indices_parametros = sorted(set(indices_analisis + [
    min(con_coincidencias, key=lambda i: longitudes[i]),
    max(con_coincidencias, key=lambda i: longitudes[i])
]), key=lambda i: doc_ids[i])
configuraciones = [('Base', 1.2, 0.75), ('Sin ajuste por longitud', 1.2, 0.0),
                   ('Mayor efecto de las repeticiones', 2.0, 0.75)]
filas_posiciones, filas_aportes = [], []
scores_configuraciones = {}
for nombre, k1, b in configuraciones:
    scores = calcular_bm25(tokens_parametros, k1=k1, b=b)
    scores_configuraciones[nombre] = scores
    # Ranking completo de documentos positivos para registrar posiciones reales.
    posiciones = {r['doc_id']: r['Posición'] for r in ranking(scores, limite=N)}
    for i in indices_parametros:
        filas_posiciones.append({'doc_id': doc_ids[i], '|d|': int(longitudes[i]),
                                  '|d| / avgdl': longitudes[i] / avgdl,
                                  'Configuración': nombre, 'k1': k1, 'b': b,
                                  'Posición': posiciones[doc_ids[i]], 'Score BM25': scores[i]})
        aportes = aportes_bm25(tokens_parametros, i, k1=k1, b=b)
        np.testing.assert_allclose(aportes['Aporte BM25'].sum(), scores[i])
        aportes.insert(0, 'Configuración', nombre)
        aportes.insert(1, 'doc_id', doc_ids[i])
        filas_aportes.append(aportes)
tabla_parametros = pd.DataFrame(filas_posiciones)
tabla_aportes_parametros = pd.concat(filas_aportes, ignore_index=True)
display(Markdown(f'**Consulta fija: {consulta_analisis}. Media: {avgdl:,.2f} tokens.**'))
display(tabla_parametros)
display(tabla_parametros.pivot(index='doc_id', columns='Configuración', values='Posición'))
display(tabla_aportes_parametros)

for i in indices_parametros:
    filas = tabla_parametros[tabla_parametros['doc_id'] == doc_ids[i]].set_index('Configuración')
    base = filas.loc['Base']
    sin_longitud = filas.loc['Sin ajuste por longitud']
    repeticion = filas.loc['Mayor efecto de las repeticiones']
    relacion = 'más corto' if longitudes[i] < avgdl else 'más largo'
    display(Markdown(
        f"- **{doc_ids[i]}**, {relacion} que la media: con b=0 pasa de la posición "
        f"{int(base['Posición'])} a {int(sin_longitud['Posición'])}, y su score cambia "
        f"de {base['Score BM25']:.6f} a {sin_longitud['Score BM25']:.6f}. "
        f"Con k₁=2.0 queda en la posición {int(repeticion['Posición'])} "
        f"con score {repeticion['Score BM25']:.6f}."
    ))
    terminos_comparados = {}
    for nombre, k1, b in configuraciones:
        terminos_comparados[nombre] = aportes_bm25(tokens_parametros, i, k1, b).set_index('Término')
    for nombre in ['Sin ajuste por longitud', 'Mayor efecto de las repeticiones']:
        diferencia = (terminos_comparados[nombre]['Aporte BM25']
                      - terminos_comparados['Base']['Aporte BM25'])
        t = diferencia.abs().idxmax()
        display(Markdown(
            f"  En **{nombre}**, el mayor cambio de aporte corresponde a `{t}` "
            f"(f={tf_documentos[i][t]}): "
            f"{terminos_comparados['Base'].loc[t, 'Aporte BM25']:.6f} → "
            f"{terminos_comparados[nombre].loc[t, 'Aporte BM25']:.6f} "
            f"(diferencia {diferencia[t]:+.6f})."
        ))
display(Markdown(
    '**Efecto de b.** Al fijarlo en cero el ajuste de longitud se vuelve 1 para todos ' 
    'los libros. Los cortos pierden la ventaja de un denominador reducido y los largos ' 
    'dejan de recibir la penalización por longitud. Las frecuencias y los IDF permanecen iguales.\n\n'
    '**Efecto de k₁.** Pasar de 1.2 a 2.0 eleva el límite del factor TF de 2.2 a 3.0 ' 
    'y hace más gradual la saturación. Los términos con frecuencias distintas cambian ' 
    'sus aportes de forma distinta, como muestra la tabla.\n\n'
    '**Posiciones estables.** Si un documento conserva su posición aun cuando cambia ' 
    'su puntuación, la suma de los nuevos aportes no cruza los scores de sus vecinos. ' 
    'Un cambio de parámetro puede modificar el margen entre documentos sin invertir su orden.'
))
assert estadisticas_antes[0] == dict(df_corpus)
assert estadisticas_antes[1] == dict(idf_bm25)
np.testing.assert_array_equal(estadisticas_antes[2], longitudes)
print('Verificación final: mismo corpus, tokens y estadísticas en todas las configuraciones.')

**Consulta fija: Q2. Media: 118,671.06 tokens.**

,doc_id,|d|,|d| / avgdl,Configuración,k1,b,Posición,Score BM25
0,bible-kjv.txt,791842,6.672579,Base,1.2,0.75,4,1.845840
1,blake-poems.txt,6934,0.058430,Base,1.2,0.75,12,1.123575
2,edgeworth-parents.txt,170786,1.439155,Base,1.2,0.75,10,1.161775
3,bible-kjv.txt,791842,6.672579,Sin ajuste por longitud,1.2,0.00,4,2.057220
4,blake-poems.txt,6934,0.058430,Sin ajuste por longitud,1.2,0.00,13,0.848830
5,edgeworth-parents.txt,170786,1.439155,Sin ajuste por longitud,1.2,0.00,10,1.170939
6,bible-kjv.txt,791842,6.672579,Mayor efecto de las repeticiones,2.0,0.75,4,2.344170
7,blake-poems.txt,6934,0.058430,Mayor efecto de las repeticiones,2.0,0.75,12,1.389857
8,edgeworth-parents.txt,170786,1.439155,Mayor efecto de las repeticiones,2.0,0.75,10,1.551653


Configuración,Base,Mayor efecto de las repeticiones,Sin ajuste por longitud
doc_id,,,
bible-kjv.txt,4,4,4
blake-poems.txt,12,12,13
edgeworth-parents.txt,10,10,10


,Configuración,doc_id,Término,"f(t,d)",df(t),IDF BM25,Ajuste de longitud,Factor TF saturado,Aporte BM25
0,Base,bible-kjv.txt,family,123,13,0.341749,5.254434,2.092721,0.715186
1,Base,bible-kjv.txt,love,311,15,0.203599,5.254434,2.156283,0.439017
2,Base,bible-kjv.txt,marriage,19,12,0.418710,5.254434,1.651826,0.691637
3,Base,blake-poems.txt,family,0,13,0.341749,0.293823,0.000000,0.000000
4,Base,blake-poems.txt,love,29,15,0.203599,0.293823,2.173573,0.442537
5,Base,blake-poems.txt,marriage,1,12,0.418710,0.293823,1.626512,0.681038
6,Base,edgeworth-parents.txt,family,39,13,0.341749,1.329366,2.113548,0.722304
7,Base,edgeworth-parents.txt,love,83,15,0.203599,1.329366,2.158514,0.439471
8,Base,edgeworth-parents.txt,marriage,0,12,0.418710,1.329366,0.000000,0.000000
9,Sin ajuste por longitud,bible-kjv.txt,family,123,13,0.341749,1.000000,2.178744,0.744584


- **bible-kjv.txt**, más largo que la media: con b=0 pasa de la posición 4 a 4, y su score cambia de 1.845840 a 2.057220. Con k₁=2.0 queda en la posición 4 con score 2.344170.

  En **Sin ajuste por longitud**, el mayor cambio de aporte corresponde a `marriage` (f=19): 0.691637 → 0.866440 (diferencia +0.174803).

... (5 outputs left out)

  En **Sin ajuste por longitud**, el mayor cambio de aporte corresponde a `family` (f=39): 0.722304 → 0.729405 (diferencia +0.007102).

  En **Mayor efecto de las repeticiones**, el mayor cambio de aporte corresponde a `family` (f=39): 0.722304 → 0.959815 (diferencia +0.237511).

**Efecto de b.** Al fijarlo en cero el ajuste de longitud se vuelve 1 para todos los libros. Los cortos pierden la ventaja de un denominador reducido y los largos dejan de recibir la penalización por longitud. Las frecuencias y los IDF permanecen iguales.

**Efecto de k₁.** Pasar de 1.2 a 2.0 eleva el límite del factor TF de 2.2 a 3.0 y hace más gradual la saturación. Los términos con frecuencias distintas cambian sus aportes de forma distinta, como muestra la tabla.

**Posiciones estables.** Si un documento conserva su posición aun cuando cambia su puntuación, la suma de los nuevos aportes no cruza los scores de sus vecinos. Un cambio de parámetro puede modificar el margen entre documentos sin invertir su orden.

Verificación final: mismo corpus, tokens y estadísticas en todas las configuraciones.
